<a href="https://colab.research.google.com/github/azrapatvi/nlp_practice/blob/main/13_emotion_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("praveengovi/emotions-dataset-for-nlp")

print("Path to dataset files:", path)

100%|██████████| 721k/721k [00:00<00:00, 91.6MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/praveengovi/emotions-dataset-for-nlp/versions/1


In [ ]:
path

'/root/.cache/kagglehub/datasets/praveengovi/emotions-dataset-for-nlp/versions/1'

In [ ]:
import os


In [ ]:
df=pd.read_csv(os.path.join(path,'train.txt'),sep=';',names=['text','emotion'])
df

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [ ]:
df.shape

(16000, 2)

In [ ]:
df['text']=df['text'].str.lower()

In [ ]:
import string # python has built librayr that has some predefined puncutation marks in it

print(string.punctuation)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [ ]:
import re

df['text']=df['text'].apply(lambda x: re.sub('[^a-zA-Z]',' ',x))

In [ ]:
df

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [ ]:
# now the text is cleaned

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk

nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from nltk.corpus import stopwords

stop_words=set(stopwords.words('english'))

In [ ]:
from nltk.stem import PorterStemmer

stemmer=PorterStemmer()

In [ ]:
def remove_stopwords(text):
  words=word_tokenize(text)

  corpus=[]
  for i in words:
    if i not in stop_words:
      corpus.append(i)

  corpus=' '.join(corpus)

  return corpus



In [ ]:
df['text']=df['text'].apply(remove_stopwords)

In [ ]:
df['text'][0]

'didnt feel humiliated'

In [ ]:
df['emotion'].value_counts()

,count
emotion,
joy,5362
sadness,4666
anger,2159
fear,1937
love,1304
surprise,572


In [ ]:
# split the data

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(df['text'],df['emotion'],test_size=0.25,random_state=43,stratify=df['emotion'])


In [ ]:
# will apply the vectorizer to convert the data into numeric
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf=TfidfVectorizer()

In [ ]:
X_train=tfidf.fit_transform(X_train)
X_test=tfidf.transform(X_test)

In [ ]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 108529 stored elements and shape (12000, 12879)>

In [ ]:
# training a model

from sklearn.naive_bayes import MultinomialNB

model=MultinomialNB()

In [ ]:
model

MultinomialNB()

In [ ]:
model.fit(X_train,y_train)

MultinomialNB()

In [ ]:
y_pred=model.predict(X_test)
y_pred

array(['sadness', 'sadness', 'joy', ..., 'joy', 'joy', 'sadness'],
      dtype='<U8')

In [ ]:
from sklearn.metrics import accuracy_score,classification_report, confusion_matrix,precision_score

print(accuracy_score(y_test, y_pred))
print(precision_score(y_test, y_pred, average='weighted'))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.66325
0.7160864828036387
              precision    recall  f1-score   support

       anger       0.95      0.30      0.46       540
        fear       0.85      0.20      0.32       484
         joy       0.62      0.96      0.75      1341
        love       1.00      0.05      0.10       326
     sadness       0.67      0.93      0.78      1166
    surprise       0.00      0.00      0.00       143

    accuracy                           0.66      4000
   macro avg       0.68      0.41      0.40      4000
weighted avg       0.72      0.66      0.59      4000

[[ 162    3  190    0  185    0]
 [   6   95  195    0  188    0]
 [   0    0 1294    0   47    0]
 [   3    1  249   17   56    0]
 [   0    0   81    0 1085    0]
 [   0   13   83    0   47    0]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/m

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train
)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

new_model=MultinomialNB()

new_model.fit(X_train,y_train,sample_weight=sample_weights)

y_pred=new_model.predict(X_test)

print(accuracy_score(y_test, y_pred))
print(precision_score(y_test, y_pred, average='weighted'))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.8255
0.8551409350408181
              precision    recall  f1-score   support

       anger       0.83      0.86      0.84       540
        fear       0.78      0.84      0.81       484
         joy       0.95      0.78      0.86      1341
        love       0.61      0.86      0.71       326
     sadness       0.91      0.84      0.87      1166
    surprise       0.44      0.85      0.58       143

    accuracy                           0.83      4000
   macro avg       0.75      0.84      0.78      4000
weighted avg       0.86      0.83      0.83      4000

[[ 464   26   11   10   18   11]
 [  17  408    7    5   16   31]
 [  24   33 1051  131   48   54]
 [   4    8   24  281    5    4]
 [  53   37   17   30  976   53]
 [   0   10    1    6    4  122]]


In [ ]:
import joblib

joblib.dump(new_model,'model.pkl',compress=3)

['model.pkl']

In [ ]:
new_data="i am very tired"

new_data=remove_stopwords(new_data)
new_data = tfidf.transform([new_data])

new_data_pred=new_model.predict(new_data)

new_data_pred

array(['sadness'], dtype='<U8')

In [ ]:
joblib.dump(tfidf, 'tfidf.pkl',compress=3)

['tfidf.pkl']